# Download OPP-115 dataset and map annotations to good/neutral/bad

In [ ]:
# Install the Hugging Face library
!pip install datasets -q

import pandas as pd
from datasets import load_dataset

In [ ]:
# 1. Fetching dataset directly from Hugging Face
hf_dataset = load_dataset("alzoubi36/opp_115", split="train")

# Convert it into a Pandas table so we can work with it
df = pd.DataFrame(hf_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/451 [00:00<?, ?B/s]

data/train-00000-of-00001-58a17c1d9a42fb(…):   0%|          | 0.00/513k [00:00<?, ?B/s]

data/validation-00000-of-00001-a2d185c06(…):   0%|          | 0.00/137k [00:00<?, ?B/s]

data/test-00000-of-00001-e939d436d2b7320(…):   0%|          | 0.00/161k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2185 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/550 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/697 [00:00<?, ? examples/s]

In [ ]:
# 2. Mapping numerical labels to Risk Categories
# Based on the hugging face OPP-115 schema, here is what those numbers mean:
# We map these numbers directly to our 3 risk levels:
hf_mapping = {
    0: 'neutral',  # Data Retention
    1: 'good',     # Data Security
    2: 'neutral',  # Do Not Track
    3: 'bad',      # First Party Collection/Use
    4: 'neutral',  # International and Specific Audiences
    5: 'neutral',  # Other / Introductory
    6: 'neutral',  # Policy Change
    7: 'neutral',  # Practice not covered
    8: 'neutral',  # Privacy contact information
    9: 'bad',      # Third Party Sharing/Collection
    10: 'good',    # User Access, Edit and Deletion
    11: 'good'     # User Choice/Control
}

In [ ]:
# 3. Create a logic function to handle the lists
def determine_highest_risk(label_list):
    # Convert the list of numbers into a list of our risk words
    risks = [hf_mapping.get(num, 'neutral') for num in label_list]

    # Prioritize 'bad' -> then 'good' -> default to 'neutral'
    if 'bad' in risks:
        return 'bad'
    elif 'good' in risks:
        return 'good'
    else:
        return 'neutral'

# Apply our custom function to every row
df['label'] = df['label'].apply(determine_highest_risk)

# Standardize columns
df['original_label_list'] = hf_dataset['label'] # Keep the original list just in case
df['clause_text'] = df['text']
df['dataset'] = 'OPP-115'

In [ ]:
# 4. Clean up the table
final_df = df[['clause_text', 'original_label_list', 'label', 'dataset']]
final_df = final_df.dropna(subset=['label'])

In [ ]:
# 5. Export the final file
final_df.to_csv('opp115_mapped_hf.csv', index=False)

print("\n--- Pipeline Execution Successful ---")
print(f"Total clauses processed: {len(final_df)}")
print("\nClass Breakdown:")
print(final_df['label'].value_counts())


--- Pipeline Execution Successful ---
Total clauses processed: 2185

Class Breakdown:
label
bad        1183
neutral     608
good        394
Name: count, dtype: int64


In [ ]:
from google.colab import files

files.download('opp115_mapped_hf.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>